# The aggregate order book

Three things, in order: what the book does, that the implementations agree on what it
does, and what the faster ones cost.

The apparatus is `documentation/order-driven-markets-notation.md` for the symbols and
`documentation/order-flow-to-order-book.md` for the results. Nothing here re-derives them.

In [ ]:
from unito26.lob.benchmark import best_price_share, measure_memory, session, time_variants
from unito26.lob import config
from unito26.lob.messages import BUY, SELL, GridDepth, limit_order, market_order
from unito26.lob.orderbook import AXIS_B_VARIANTS, AggregateBook
from unito26.lob.simulate import MarkParams
from unito26.lob.visualization import ascii_ladder, book_figure, depth_figure, example_figure
from unito26.lob.worked_examples import CATALOGUE, SECTION_8_BOOK, check, reflect, to_sides

import pandas as pd

In [ ]:
%matplotlib inline

## 1. What the book does

The book of section 8 of the notation, and two readings of it: the ladder, which is the
book as a trader sees it, and the depth curve, which is the same data as an execution
cost. The slope of the second measures the book's resilience to size.

In [ ]:
book = AggregateBook.from_levels(*to_sides(SECTION_8_BOOK))
print(ascii_ladder(book))
print()
print(f"P^b = {book.best_bid_price}, P^a = {book.best_ask_price}, "
      f"phi = {book.spread}, P^m = {book.mid_price}, I^1 = {book.queue_imbalance(GridDepth(1)):+.4f}")

In [ ]:
book_figure(book, depth=6, title='the section 8 book')

In [ ]:
depth_figure(book, depth=6)

### One order, before and after

Case B is canonical because it exercises several branches of `prop.lobUpdate` at once:
the order walks the bid side, the remainder rests *inside* the old spread, every ask
index shifts, and two grid positions between the new best ask and the old one hold
nothing at all.

In [ ]:
case_b = next(e for e in CATALOGUE if "case B" in e.name)
example_figure(case_b)

In [ ]:
after = AggregateBook.from_levels(*to_sides(case_b.before))
result = after.submit(case_b.message, record=True)
print(f"q_M = {result.market_order_size}, walked = {result.walked_the_book}, "
      f"unfilled = {result.unfilled}")
print(f"fills at the RESTING prices: {[(f.price, f.size) for f in result.fills]}")
print(f"the two empty levels: {after.levels(SELL, 4)}")

### Market-to-limit

A market order names no price, so its remainder rests at the price it last executed
against. The one remainder that cannot rest is one that executed nothing — there is no
price to inherit — and those shares are reported rather than dropped.

In [ ]:
walked = AggregateBook.from_levels(*to_sides(SECTION_8_BOOK))
big = walked.submit(market_order(1.0, 1000, SELL), record=True)
print(f"traded {big.market_order_size}, unfilled {big.unfilled}; the rest is now an ask:")
print(ascii_ladder(walked))

empty = AggregateBook.from_levels({1000: 100}, {})
nothing = empty.submit(market_order(1.0, 60, BUY), record=True)
print(f"\ninto an empty ask side: {nothing.market_order_size} traded, "
      f"{nothing.unfilled} unfilled, book unchanged")

## 2. That the implementations agree

The catalogue holds one example per branch of the update rule, with the expected state
derived from the notation rather than captured from a run: a fixture recorded by running
the code certifies only that the code has not changed.

Every variant is run through the catalogue and through its mirror, the mirror being prices
reflected and every direction flipped. That the mirror passes is the test of the claim that
$d$ collapses both sides of the book into one comparison.

In [ ]:
rows = []
for cls in AXIS_B_VARIANTS:
    for example in CATALOGUE:
        for case in (example, reflect(example, 1001)):
            try:
                check(cls, case)
                rows.append((cls.__name__, case.name, "pass"))
            except AssertionError as failure:
                rows.append((cls.__name__, case.name, str(failure)))
results = pd.DataFrame(rows, columns=["variant", "example", "outcome"])
print(results.outcome.value_counts().to_string())
results.pivot_table(index="example", columns="variant", values="outcome", aggfunc="first").head(12)

## 3. What the faster ones cost

Two regimes, from the same simulator with different `depth_decay`: one where flow
concentrates at the touch and few levels are ever occupied, and one where it spreads. The
ladder's answer is conditional on which of the two the market is in.

In [ ]:
shallow = session("shallow", config.shallow_mark_params(), horizon=3600.0, seed=0)
deep = session("deep", config.deep_mark_params(), horizon=3600.0, seed=0)
for run_ in (shallow, deep):
    print(f"{run_.name}: {len(run_.messages)} messages, L = {run_.occupied_levels()}")

In [ ]:
timings = {run_.name: time_variants(AXIS_B_VARIANTS, run_, repeat=3) for run_ in (shallow, deep)}
table = pd.DataFrame(timings)
speedup = table.loc["AggregateBook"] / table
speedup.columns = [f"{name} speedup" for name in speedup.columns]
pd.concat([(table * 1000).round(1).add_suffix(" ms"), speedup.round(2)], axis=1)

The share of run time the baseline spends looking up its best price bounds what the ladder
can win: nothing else in it is changed, so this is the ceiling on every speedup above.

It must be measured as *cumulative* time. `best_price` scans by calling `max`, and a
profiler bills a builtin to itself, so the method's own time reads about 3% where the
cumulative time is thirty.

In [ ]:
pd.Series(
    {run_.name: best_price_share(AggregateBook, run_) for run_ in (shallow, deep)},
    name="best_price share of run time",
).map("{:.1%}".format)

### Memory, which tells a different story

Time ranks the variants closely; space does not. The separation is where two of the
techniques are paid for: lazy deletion accumulates stale heap entries in proportion to
every level that has ever existed, and a tick-indexed array costs its whole band whether
or not the levels are occupied.

In [ ]:
memory = {
    run_.name: {name: sizes["resident"] / 1024 for name, sizes in
                measure_memory(AXIS_B_VARIANTS, run_).items()}
    for run_ in (shallow, deep)
}
pd.DataFrame(memory).round(1).add_suffix(" kB resident")

The two tables are read together. `HeapBook` is competitive on time and a hundred times
the size of the book it indexes, until `compact` is called. `TickArrayBook` is fastest in
the deep regime, and smallest here only because the band happens to be narrow: on a wide,
sparse instrument the same code would be the largest by far. Neither fact is visible in
the timings, so a choice made on the timings alone is made without them.